# 02 — Feature Engineering

Подготовка ML-датасета из cleaned EDA-данных.

**Input:** raw datasets (читаем заново, применяем cleaning из `utils/cleaning.py`)

**Output:** `data/processed/X_train.parquet`, `X_val.parquet`, `X_test.parquet`, `y_*.parquet`, `review_embeddings.npy`

## Setup

Весь cleaning описан в `utils/cleaning.py` — вызываем одной строкой `load_all()`.
Notebooks 03, 04, 05 читают артефакты из `data/processed/`.

In [1]:
# Базовые библиотеки и пути
import os, sys, re, ast, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split

warnings.filterwarnings("ignore")
sys.path.insert(0, "..")
from utils.cleaning import load_all

# GPU-ускорение (опционально)
try:
    import cudf
    HAS_CUDF = True
except ImportError:
    HAS_CUDF = False
try:
    import torch
    HAS_CUDA = torch.cuda.is_available()
except ImportError:
    HAS_CUDA = False

# NLP (опционально)
try:
    from transformers import pipeline
    HAS_TRANSFORMERS = True
except ImportError:
    HAS_TRANSFORMERS = False
try:
    from sentence_transformers import SentenceTransformer
    HAS_ST = True
except ImportError:
    HAS_ST = False

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", None)

RAW_DIR       = Path("../data/raw")
PROCESSED_DIR = Path("../data/processed")
MODELS_DIR    = Path("../models")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"GPU CUDA : {HAS_CUDA}   cuDF : {HAS_CUDF}")
print(f"NLP      : Transformers={HAS_TRANSFORMERS}   SentenceTransformers={HAS_ST}")

GPU CUDA : True   cuDF : True
NLP      : Transformers=True   SentenceTransformers=True


In [2]:
# Загружаем все 4 датасета (cleaning внутри utils/cleaning.py)
data = load_all(data_dir=str(RAW_DIR))
listings = data["listings"]
calendar = data["calendar"]
reviews  = data["reviews"]
geo      = data["geo"]
print(f"listings : {listings.shape}")
print(f"calendar : {calendar.shape}")
print(f"reviews  : {reviews.shape}")
print(f"geo      : {geo.shape}")
already = ["log_price", "distance_to_center", "unavailability_rate",
           "calendar_unavail_rate", "host_experience_years", "professional_host"]
print("Already done by cleaning:", [c for c in already if c in listings.columns])

listings : (10448, 83)
calendar : (3825200, 9)
reviews  : (501053, 6)
geo      : (22, 2)
Already done by cleaning: ['log_price', 'distance_to_center', 'unavailability_rate', 'calendar_unavail_rate', 'host_experience_years', 'professional_host']


## Удаление leakage-колонок

Убираем признаки, которые косвенно содержат target или тавтологичны уже созданному `unavailability_rate`:

- `estimated_revenue_l365d`, `estimated_occupancy_l365d` — рассчитаны Airbnb из той же цены (post-booking).
- `availability_30/60/90/365`, `availability_eoy`, `has_availability` — дублируют `unavailability_rate`.

In [3]:
leakage_cols = [
    "estimated_revenue_l365d", "estimated_occupancy_l365d",
    "availability_30", "availability_60", "availability_90",
    "availability_365", "availability_eoy", "has_availability",
]
listings = listings.drop(columns=[c for c in leakage_cols if c in listings.columns])
print(f'shape после удаления утечек: {listings.shape}')

shape после удаления утечек: (10448, 75)


**Зачем убирать -** 

 `estimated_revenue_l365d` и `estimated_occupancy_l365d` рассчитаны Airbnb из той же цены — модель «подсмотрела» бы target. Колонки `availability_*` дублируют уже созданный `unavailability_rate`. 
 
 Это классический target leakage — без чистки модель показала бы завышенные метрики на validation, но плохо работала бы в продакшене.


## Host features

- `host_response_rate`, `host_acceptance_rate` — парсинг `"95%"` → `0.95`
- `host_has_about` — флаг заполнения раздела «о себе»
- `host_verifications_count` — кол-во методов верификации
- `professional_host` (уже создан в cleaning) — bool → int

In [4]:
# "95%" → 0.95
for col in ["host_response_rate", "host_acceptance_rate"]:
    listings[col] = (listings[col].astype(str).str.replace("%", "", regex=False)
                     .pipe(pd.to_numeric, errors="coerce") / 100)
# Заполнил ли хост поле «о себе»
listings["host_has_about"] = (listings["host_about"].fillna("").str.strip() != "").astype(int)
# host_verifications хранится строкой вида "["email","phone","work_email"]"
def _count_verifs(s):
    try: return len(ast.literal_eval(s))
    except Exception: return 0
listings["host_verifications_count"] = listings["host_verifications"].fillna("[]").apply(_count_verifs)
# professional_host (≥3 объявлений) → int для моделей
listings["professional_host"] = listings["professional_host"].astype(int)
listings[["host_response_rate", "host_acceptance_rate",
          "host_has_about", "host_verifications_count"]].describe().round(2)

,host_response_rate,host_acceptance_rate,host_has_about,host_verifications_count
count,6658.00,8026.00,10448.00,10448.00
mean,0.92,0.71,0.52,2.01
std,0.21,0.32,0.50,0.44
min,0.00,0.00,0.00,0.00
25%,1.00,0.50,0.00,2.00
50%,1.00,0.83,1.00,2.00
75%,1.00,1.00,1.00,2.00
max,1.00,1.00,1.00,3.00


## Demand features (calendar)

Сигналы спроса из календаря на 365 дней вперёд (3.8M строк):
- `unavail_summer` / `unavail_winter` / `unavail_other` — сезонная занятость
- `unavail_weekend_diff` — weekend-эффект (выходные минус будни)
- `avg_min_nights` — средний `minimum_nights` по календарю

In [5]:
# cuDF (GPU pandas) ускоряет тяжёлый groupby. Если не установлен — fallback на pandas.
if HAS_CUDF:
    cal = cudf.from_pandas(calendar)
    engine = "GPU (cuDF)"
else:
    cal = calendar.copy()
    engine = "CPU (pandas)"
print(f'[{engine}] строк: {len(cal):,}')
cal["unavail"] = 1 - cal["available"]
season_map = {6:"summer", 7:"summer", 8:"summer", 12:"winter", 1:"winter", 2:"winter"}
cal["season"] = cal["month"].map(season_map).fillna("other")
# Хэлпер: переносим результат groupby в pandas (unstack/pivot в cuDF неполный)
def _to_pd(obj):
    return obj.to_pandas() if hasattr(obj, "to_pandas") else obj
season_unavail = _to_pd(cal.groupby(["listing_id", "season"])["unavail"].mean()).unstack().add_prefix("unavail_")
wknd = _to_pd(cal.groupby(["listing_id", "is_weekend"])["unavail"].mean()).unstack()
wknd.columns = ["unavail_weekday", "unavail_weekend"]
wknd["unavail_weekend_diff"] = wknd["unavail_weekend"] - wknd["unavail_weekday"]
avg_min = _to_pd(cal.groupby("listing_id")["minimum_nights"].mean()).rename("avg_min_nights")
demand = season_unavail.join(wknd[["unavail_weekend_diff"]]).join(avg_min)
listings = listings.merge(demand, left_on="id", right_index=True, how="left")
print("новые demand-фичи:", demand.columns.tolist())

[GPU (cuDF)] строк: 3,825,200


новые demand-фичи: ['unavail_winter', 'unavail_other', 'unavail_summer', 'unavail_weekend_diff', 'avg_min_nights']


**Зачем эти фичи.** Сезонные и weekend-метрики показывают модели не только «сколько объект был занят в среднем», но и **как** он был занят: летние пики, weekend-эффект. Этого нет в `unavailability_rate` — там только годовое среднее.


## Geo features

- `geo_cluster` — 5 пространственных KMeans-кластеров по `(lat, lon)`
- `neighborhood_listing_density` — кол-во объявлений в районе (не зависит от цены, leakage нет)
- `is_tourist_zone` — менее 2 км от центра

In [6]:
listings["geo_cluster"] = KMeans(n_clusters=5, random_state=SEED, n_init=10).fit_predict(
    listings[["latitude", "longitude"]]
)
nb_agg = listings.groupby("neighbourhood_cleansed").agg(
    neighborhood_listing_density=("id", "count"),
)
listings = listings.merge(nb_agg, left_on="neighbourhood_cleansed", right_index=True, how="left")
listings["is_tourist_zone"] = (listings["distance_to_center"] < 2).astype(int)
listings[["geo_cluster", "neighborhood_listing_density", "is_tourist_zone"]].head()

,geo_cluster,neighborhood_listing_density,is_tourist_zone
0,2,1202,0
1,2,1202,1
2,3,920,1
3,1,920,1
4,3,119,0


## Amenities features

Парсинг JSON-строки `amenities` → 12 бинарных флагов + общий счётчик + premium-score.

In [7]:
def _parse_amen(s):
    try: return [a.lower() for a in ast.literal_eval(s)]
    except Exception: return []
amen = listings["amenities"].fillna("[]").apply(_parse_amen)
listings["amenities_count"] = amen.apply(len)
amenity_flags = {
    "has_wifi":         ["wifi"],
    "has_kitchen":      ["kitchen"],
    "has_parking":      ["parking"],
    "has_workspace":    ["workspace", "desk"],
    "has_ac":           ["air conditioning"],
    "has_washer":       ["washer"],
    "has_dishwasher":   ["dishwasher"],
    "has_balcony":      ["balcony", "patio"],
    "has_self_checkin": ["self check-in", "self check in"],
    "has_pool":         ["pool"],
    "has_gym":          ["gym"],
    "has_elevator":     ["elevator"],
}
for flag, keys in amenity_flags.items():
    listings[flag] = amen.apply(lambda lst: int(any(k in a for a in lst for k in keys)))
# Премиум-индекс = сумма «люксовых» удобств
premium = ["has_pool", "has_gym", "has_balcony", "has_dishwasher", "has_ac", "has_elevator"]
listings["amenities_premium_score"] = listings[premium].sum(axis=1)
listings[list(amenity_flags) + ["amenities_count", "amenities_premium_score"]].sum().sort_values(ascending=False)

amenities_count            334254
amenities_premium_score     13005
has_wifi                    10293
has_kitchen                  8998
has_washer                   8671
has_parking                  6268
has_dishwasher               5893
has_workspace                5773
has_balcony                  4582
has_self_checkin             2935
has_ac                       1406
has_elevator                  791
has_gym                       199
has_pool                      134
dtype: int64

## Property / room features

- `room_type_ord` — порядковый код (Shared=0 … Entire=3)
- `property_type_grp` — 4 группы (apartment / house / hotel / other)
- `bathrooms_n`, `bathrooms_shared` — парсинг из `bathrooms_text`

In [8]:
room_order = {"Shared room": 0, "Hotel room": 1, "Private room": 2, "Entire home/apt": 3}
listings["room_type_ord"] = listings["room_type"].map(room_order)
def _group_property(s):
    s = str(s).lower()
    if "hotel" in s or "hostel" in s: return "hotel"
    if "house" in s or "townhouse" in s or "cottage" in s: return "house"
    if "apart" in s or "condo" in s or "loft" in s or "rental unit" in s: return "apartment"
    return "other"
listings["property_type_grp"] = listings["property_type"].apply(_group_property)
def _parse_bath(s):
    if pd.isna(s): return np.nan, 0
    s = str(s).lower()
    m = re.search(r'(\d+\.?\d*)', s)
    n = float(m.group(1)) if m else (0.5 if "half" in s else np.nan)
    return n, int("shared" in s)
listings[["bathrooms_n", "bathrooms_shared"]] = listings["bathrooms_text"].apply(
    lambda s: pd.Series(_parse_bath(s))
)
print("property_type_grp:", listings["property_type_grp"].value_counts().to_dict())
listings[["room_type_ord", "property_type_grp", "bathrooms_n", "bathrooms_shared"]].head()

property_type_grp: {'apartment': 7359, 'other': 1992, 'house': 645, 'hotel': 452}


,room_type_ord,property_type_grp,bathrooms_n,bathrooms_shared
0,2,house,1.5,0.0
1,2,apartment,1.0,1.0
2,2,apartment,1.0,1.0
3,3,apartment,1.5,0.0
4,3,apartment,1.5,0.0


## NLP features

### Text features
Длины полей `name` / `description` и наличие маркетинговых ключевых слов (luxury / central / spacious / cozy / view).

### Sentiment (BERT multilingual)
`nlptown/bert-base-multilingual-uncased-sentiment` → шкала 1–5 звёзд, нормализуем в `[-1, +1]`.
Берём 5 последних отзывов на listing.

### Review embeddings (для recommender в nb05)
`sentence-transformers/all-MiniLM-L6-v2` → 384-d вектор на listing.
Сохраняем в `data/processed/review_embeddings.npy`.

In [9]:
listings["name_length"]        = listings["name"].fillna("").str.len()
listings["description_length"] = listings["description"].fillna("").str.len()
keywords = ["luxury", "central", "spacious", "cozy", "view"]
text = (listings["name"].fillna("") + " " + listings["description"].fillna("")).str.lower()
listings["description_has_keywords"] = text.apply(lambda t: int(any(k in t for k in keywords)))
listings[["name_length", "description_length", "description_has_keywords"]].describe().round(1)

,name_length,description_length,description_has_keywords
count,10448.0,10448.0,10448.0
mean,37.1,385.0,0.7
std,10.0,144.3,0.5
min,1.0,0.0,0.0
25%,30.0,299.0,0.0
50%,38.0,443.0,1.0
75%,47.0,497.0,1.0
max,66.0,1000.0,1.0


In [10]:
# 5 последних отзывов на listing — иначе 500K вызовов BERT
recent = (reviews.sort_values("date").groupby("listing_id").tail(5)).copy()
if HAS_TRANSFORMERS:
    device = 0 if HAS_CUDA else -1
    print(f'[BERT] device = {"GPU (cuda:0)" if device == 0 else "CPU"}')
    bert = pipeline("sentiment-analysis",
                    model="nlptown/bert-base-multilingual-uncased-sentiment",
                    truncation=True, max_length=256,
                    device=device, batch_size=64)
    texts = recent["comments"].astype(str).str[:500].tolist()
    outs  = bert(texts)
    # label="1 star".."5 stars" → (звёзды-3)/2 ∈ [-1, +1]
    recent["bert"] = [(int(o["label"][0]) - 3) / 2 for o in outs]
    bert_agg = recent.groupby("listing_id")["bert"].mean().rename("reviews_sentiment_transformer")
    listings = listings.merge(bert_agg, left_on="id", right_index=True, how="left")
    print("reviews_sentiment_transformer:", listings["reviews_sentiment_transformer"].describe().round(3).to_dict())
else:
    print("⚠ transformers/torch не установлены — раздел пропущен")

[BERT] device = GPU (cuda:0)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

reviews_sentiment_transformer: {'count': 9362.0, 'mean': 0.844, 'std': 0.186, 'min': -1.0, '25%': 0.8, '50%': 0.9, '75%': 1.0, 'max': 1.0}


In [11]:
# Эмбеддинги отзывов — векторное представление для recommender (nb05)
# MiniLM = маленькая sentence-transformer модель (384 dim, быстро)
if HAS_ST:
    device = "cuda" if HAS_CUDA else "cpu"
    print(f'[MiniLM] device = {device}')
    model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2", device=device)
    # 5 последних отзывов → один текст на listing → один вектор
    listing_texts = (recent.groupby("listing_id")["comments"]
                     .apply(lambda x: " ".join(x.astype(str).str[:200])))
    embeddings = model.encode(listing_texts.tolist(), batch_size=128,
                              show_progress_bar=True, convert_to_numpy=True)
    np.save(PROCESSED_DIR / "review_embeddings.npy", embeddings)
    pd.DataFrame({"listing_id": listing_texts.index}).to_parquet(
        PROCESSED_DIR / "review_embeddings_ids.parquet"
    )
    print(f'embeddings: {embeddings.shape} → review_embeddings.npy')
else:
    print("⚠ sentence-transformers не установлен — эмбеддинги пропущены")

[MiniLM] device = cuda


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/74 [00:00<?, ?it/s]

embeddings: (9383, 384) → review_embeddings.npy


## Подготовка ML-датасета

- Train / val / test = 70 / 15 / 15, стратификация по `neighbourhood_cleansed`
- Категориальные строки (`neighbourhood_cleansed`, `property_type_grp`) оставляем — CatBoost обработает сам, для linear-моделей в nb03 применим one-hot
- Артефакты сохраняем в `data/processed/`

In [12]:
# Только объекты с известной ценой (target = log_price)
ml = listings[listings["price"].notna()].copy()
# id/URL/текст/сырые строки и target — НЕ идут в фичи
drop_cols = [
    "id", "listing_url", "scrape_id", "source", "name", "description",
    "neighborhood_overview", "picture_url", "host_id", "host_url", "host_name",
    "host_thumbnail_url", "host_picture_url", "host_neighbourhood", "host_location",
    "host_about", "host_verifications", "host_response_time",
    "last_scraped", "calendar_last_scraped", "first_review", "last_review", "host_since",
    "license", "bathrooms_text", "amenities", "neighbourhood",
    "property_type", "room_type",   # оставляем закодированные версии
    "price",                         # target — отдельно
]
target_col = "log_price"
X = ml.drop(columns=[c for c in drop_cols if c in ml.columns] + [target_col])
y = ml[target_col]
strat = ml["neighbourhood_cleansed"]
# 70 / 30 → затем 30 → 15 / 15
X_tr, X_tmp, y_tr, y_tmp, s_tr, s_tmp = train_test_split(
    X, y, strat, test_size=0.30, random_state=SEED, stratify=strat
)
X_val, X_te, y_val, y_te = train_test_split(
    X_tmp, y_tmp, test_size=0.50, random_state=SEED, stratify=s_tmp
)

print(f'train: {X_tr.shape}   val: {X_val.shape}   test: {X_te.shape}')
for name, df in [("X_train", X_tr), ("X_val", X_val), ("X_test", X_te)]:
    df.to_parquet(PROCESSED_DIR / f"{name}.parquet")
for name, ser in [("y_train", y_tr), ("y_val", y_val), ("y_test", y_te)]:
    ser.to_frame().to_parquet(PROCESSED_DIR / f"{name}.parquet")
ml.to_parquet(PROCESSED_DIR / "listings_features.parquet")
print(f'\nИтого фичей: {X_tr.shape[1]}')

train: (4090, 76)   val: (877, 76)   test: (877, 76)



Итого фичей: 76


**Сплит 70/15/15** со стратификацией по `neighbourhood_cleansed` — все 22 района Амстердама представлены во всех трёх выборках в нужных пропорциях. Без стратификации редкие районы могли бы целиком попасть в val или test, и модель оценивалась бы на «новых» для неё районах — это исказило бы метрики.


## Deliverable

- `../data/processed/X_train.parquet`, `X_val.parquet`, `X_test.parquet`
- `../data/processed/y_train.parquet`, `y_val.parquet`, `y_test.parquet`
- `../data/processed/listings_features.parquet` — полный feature-датасет (для nb03 / 04 / 05)
- `../data/processed/review_embeddings.npy` + `review_embeddings_ids.parquet`
- **76 фичей** в `X_*`

## Справочник фичей

Итого **76 признаков**, сгруппированных по смыслу. Курсивом — поля, созданные в `utils/cleaning.py`; остальные — в этом ноутбуке.

### 🏠 Жильё и параметры брони (13)
`accommodates`, `bathrooms`, `bedrooms`, `beds`,
`minimum_nights`, `maximum_nights`,
`minimum_minimum_nights`, `maximum_minimum_nights`,
`minimum_maximum_nights`, `maximum_maximum_nights`,
`minimum_nights_avg_ntm`, `maximum_nights_avg_ntm`,
`instant_bookable`

### 👤 Хост (15)
`host_response_rate`, `host_acceptance_rate`, `host_is_superhost`,
`host_listings_count`, `host_total_listings_count`,
`host_has_profile_pic`, `host_identity_verified`,
*`host_experience_years`*, *`professional_host`*,
`host_has_about`, `host_verifications_count`,
`calculated_host_listings_count`,
`calculated_host_listings_count_entire_homes`,
`calculated_host_listings_count_private_rooms`,
`calculated_host_listings_count_shared_rooms`

### 📍 География (7)
`latitude`, `longitude`, `neighbourhood_cleansed`,
*`distance_to_center`*, `geo_cluster`,
`neighborhood_listing_density`, `is_tourist_zone`

### 📅 Спрос из календаря (7)
*`unavailability_rate`*, *`calendar_unavail_rate`*,
`unavail_summer`, `unavail_winter`, `unavail_other`,
`unavail_weekend_diff`, `avg_min_nights`

### 🛁 Тип жилья и удобства (18)
`room_type_ord`, `property_type_grp`,
`bathrooms_n`, `bathrooms_shared`,
`amenities_count`, `amenities_premium_score`,
`has_wifi`, `has_kitchen`, `has_parking`, `has_workspace`,
`has_ac`, `has_washer`, `has_dishwasher`, `has_balcony`,
`has_self_checkin`, `has_pool`, `has_gym`, `has_elevator`

### ⭐ Отзывы и рейтинги (13)
`number_of_reviews`, `number_of_reviews_ltm`,
`number_of_reviews_l30d`, `number_of_reviews_ly`,
`reviews_per_month`,
`review_scores_rating`, `review_scores_accuracy`,
`review_scores_cleanliness`, `review_scores_checkin`,
`review_scores_communication`, `review_scores_location`,
`review_scores_value`, `reviews_sentiment_transformer`

### 📝 Текст (3)
`name_length`, `description_length`, `description_has_keywords`

**Премиум-индекс:** `amenities_premium_score = has_pool + has_gym + has_balcony + has_dishwasher + has_ac + has_elevator` (0–6).

## Итоги

В этом ноутбуке cleaned-данные превращены в ML-готовый датасет.

Что сделано:
- удалены leakage-колонки (`estimated_revenue_l365d`, `estimated_occupancy_l365d`, `availability_*`);
- хост-фичи: парсинг процентов, флаг `host_has_about`, количество verifications, `professional_host` → int;
- демогр. сигналы из календаря: сезонная и weekend-занятость, средний `min_nights`;
- гео-фичи: 5 KMeans-кластеров, плотность listings по району, флаг tourist zone;
- amenities: 12 бинарных флагов + `amenities_count` + `amenities_premium_score`;
- property/room: `room_type_ord`, `property_type_grp`, парсинг `bathrooms_text`;
- NLP: длины текстов, маркетинговые ключевые слова, BERT-sentiment по 5 последним отзывам, MiniLM-эмбеддинги для recommender;
- стратифицированный split 70/15/15 по `neighbourhood_cleansed`.

Итог: **76 фичей** в `X_train/val/test.parquet`, target `log_price` в `y_*.parquet`, эмбеддинги в `review_embeddings.npy`. Все артефакты лежат в `data/processed/` и `models/`, и используются в nb03 (pricing), nb04 (market intelligence) и nb05 (recommender).
